In [15]:
from embedder import Embedder

embed = Embedder()

q = "How does approximate nearest neighbor search work?"
v = embed.encode(q)

In [16]:
v[0]

np.float64(-0.020582036807885073)

In [17]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [18]:
d = "02-vector-search/lessons/07-sqlitesearch-vector.md"
dv = embed.encode(d)

In [19]:
v.dot(dv)

np.float64(0.3770526250449936)

In [20]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [23]:
texts = [chunk["content"] for chunk in chunks]

from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)
X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [24]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [26]:
query = "What metric do we use to evaluate a search engine?"
v_query = embed.encode(query)

In [27]:
results = vindex.search(v_query, num_results=5)
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'